# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane: **classification**. Slicing the mid-panel month `month=2026-03` from `fact_content_daily_performance`. The final month (June 2026 / `_sample`) is never used for label logic — it's kept as a sealed test month.

## Setup — connect to the warehouse (Hugging Face via DuckDB)

In [28]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb, os
import pandas as pd
from google.colab import userdata

# HF token from Colab Secrets (key icon in the left sidebar) — never hardcode it, this repo is public
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# Register the token as a DuckDB secret — without this, read_parquet('hf://...') fails with HTTP 401
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")

MONTH = "2026-03"
TABLE_URI = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet"

print("Ready. Target partition:", TABLE_URI)

Ready. Target partition: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


## 1. Unit of analysis + time window

One row (after aggregation) = one content item (`client_hash_id` + `content_hash_id`) summarized from daily performance over a single month (`month=2026-03`, 2026-03-01 through 2026-03-31).

The raw source is `fact_content_daily_performance`, whose native grain is `report_date` × `client_hash_id` × `content_hash_id` (proven in Section 3). This slice uses a mid-panel month rather than `_sample` (June 2026), so label logic is never developed inside its own outcome window.

In [29]:
con.execute(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM read_parquet('{TABLE_URI}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_d,max_d
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

- **Feature** (knowable before the month ends, safe to use): `gsc_avg_position`, `gsc_clicks`, `gsc_impressions`, CTR computed as `gsc_clicks / gsc_impressions` (no raw `ctr` column exists in this table), `ga4_data_available` (used as an availability flag, not a performance value).
- **Label / proxy**: decline in performance within the month — defined as total `gsc_clicks` in the first half (Mar 1–15) versus the second half (Mar 16–31) of the same month. This is a temporary proxy for the classification lane, to be refined in later modeling weeks.
- **Context** (for joining/splitting/grouping, never for the model to learn from): `client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_available` (used only to filter, not as a feature value).
- **Excluded**:
  - `fact_content_query_90d` — its 90-day window overlaps the final snapshot months, risking leakage if paired with the March label.
  - Rows with `ga4_data_available = FALSE` are excluded from GA4-based features because their values are zero-filled, not real observations (not "no engagement").
  - Rows with `gsc_data_available = FALSE` are excluded entirely from this slice, since GSC metrics (position, clicks, impressions) are the core features for this lane.

In [30]:
cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{TABLE_URI}') LIMIT 1").df()
print(cols["column_name"].tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify with queries, build 5 features, and the leakage trap

### 3a. Grain check — prove one raw row really is report_date × client_hash_id × content_hash_id

In [31]:
grain_check = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{TABLE_URI}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print("Duplicate rows on the grain (should be empty):")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows on the grain (should be empty):


,report_date,client_hash_id,content_hash_id,c


### 3b. Row count + date span for this slice

In [32]:
con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        MIN(report_date) AS min_d,
        MAX(report_date) AS max_d
    FROM read_parquet('{TABLE_URI}')
""").df()

,n_rows,n_content,n_clients,min_d,max_d
0,9841378,331437,55,2026-03-01,2026-03-31


### 3c. Availability check — filter with IS TRUE

In [33]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4
    FROM read_parquet('{TABLE_URI}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_gsc,rows_with_ga4
0,9841378,3611061.0,413966.0


### 3d. Build the feature frame — 5 features, content-month grain

1. `avg_position` — knowable because it's measured continuously through the month, not a future outcome.
2. `total_clicks` — knowable, a historical observation within the same window.
3. `total_impressions` — knowable, historical, never touches the future.
4. `avg_ctr` — knowable, computed from historical clicks/impressions in the same window.
5. `has_ga4_data` — knowable, an availability flag that's already certain at that moment (not a performance value).

In [34]:
feat = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks)       AS total_clicks,
        SUM(gsc_impressions)  AS total_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data
    FROM read_parquet('{TABLE_URI}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,avg_position,total_clicks,total_impressions,avg_ctr,has_ga4_data
0,content_05597932fe4da067,client_73cda7b4e4f265ea,2.714744,0.0,57.0,0.000000,0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,7.209549,7.0,6523.0,0.001073,1
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,6.481453,0.0,149.0,0.000000,1
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2.987198,0.0,453.0,0.000000,0
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,6.724039,6.0,5630.0,0.001066,1


### 3e. The leakage trap — leak on purpose, watch the score jump, then remove it

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Proxy label: second-half clicks < first-half clicks in the same month -> "declining"
label_df = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(CASE WHEN report_date <= DATE '{MONTH}-15' THEN gsc_clicks ELSE 0 END) AS clicks_h1,
        SUM(CASE WHEN report_date >  DATE '{MONTH}-15' THEN gsc_clicks ELSE 0 END) AS clicks_h2
    FROM read_parquet('{TABLE_URI}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

label_df["is_declining"] = (label_df["clicks_h2"] < label_df["clicks_h1"]).astype(int)

df = feat.merge(
    label_df[["content_hash_id", "client_hash_id", "is_declining", "clicks_h2"]],
    on=["content_hash_id", "client_hash_id"],
)
df["avg_ctr"] = df["avg_ctr"].fillna(0)

FEATURES_HONEST = ["avg_position", "total_clicks", "total_impressions", "avg_ctr", "has_ga4_data"]

# --- honest model: only the 5 knowable features ---
X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES_HONEST], df["is_declining"], test_size=0.3, random_state=42
)
model_honest = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc_honest = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print("Honest AUC:", round(auc_honest, 4))

# --- deliberate leak: add clicks_h2, which the label is directly derived from ---
df["LEAK_clicks_h2"] = df["clicks_h2"]
FEATURES_LEAK = FEATURES_HONEST + ["LEAK_clicks_h2"]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    df[FEATURES_LEAK], df["is_declining"], test_size=0.3, random_state=42
)
model_leak = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
auc_leak = roc_auc_score(y_test2, model_leak.predict_proba(X_test2)[:, 1])
print("Leaked AUC (should jump much higher, near 1.0):", round(auc_leak, 4))

# remove the leaked column — keep only the honest number
df = df.drop(columns=["LEAK_clicks_h2"])
print("\nLeak column removed. Final honest number kept: Honest AUC =", round(auc_honest, 4))

Honest AUC: 0.7534
Leaked AUC (should jump much higher, near 1.0): 1.0

Leak column removed. Final honest number kept: Honest AUC = 0.7534


## 4. Data limits

This slice covers only one month (March 2026) from clients whose GSC history already started before that date — clients with a `gsc_data_start` after March 2026 don't appear at all, so the slice is biased toward clients with longer histories. The proxy label (within-month click comparison) is also sensitive to short-term noise and hasn't been validated against a longer-horizon (90-day) outcome the way the starter dataset's label was.

In [36]:
con.execute(f"""
    SELECT client_hash_id, MIN(report_date) AS first_seen
    FROM read_parquet('{TABLE_URI}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id
    ORDER BY first_seen DESC
    LIMIT 10
""").df()

,client_hash_id,first_seen
0,client_e00b29e582949543,2026-03-27
1,client_810019792c9b8efc,2026-03-25
2,client_f6f0cdf26d03d7bd,2026-03-19
3,client_b77d0d5f08f05e64,2026-03-12
4,client_86ebc2f12c01f586,2026-03-11
5,client_a1203ffecad62470,2026-03-09
6,client_764ae36a94e30a25,2026-03-07
7,client_599043c0ff13edea,2026-03-07
8,client_cd12bcfd98942aa1,2026-03-01
9,client_f623b01661d4bfe4,2026-03-01


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.